##  Step 1: Import Necessary libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

## Step 2: CHANGE ONLY THIS SECTION

In [2]:
IMAGE_SIZE = (224, 224)          # <- Change input size
BATCH_SIZE = 32
EPOCHS = 10

TRAIN_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\train_data"
VAL_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\val_data"
TEST_DIR=r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data"
NUM_CLASSES = 5                  # <- Number of classes

BASE_MODEL_NAME ="ResNet50"      # <- Options: VGG16, ResNet50, MobileNetV2

FREEZE_LAYERS = True             # <- Freeze base model
FINE_TUNE_AT = None              # <- Set layer index to unfreeze later

## Step 3: Data Preparation || Data Pipeline

In [3]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   horizontal_flip=True,
                                   zoom_range=0.2)

val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = train_datagen.flow_from_directory(TEST_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical',
                                               shuffle=False)
train_data = train_datagen.flow_from_directory(TRAIN_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical')

val_data = val_datagen.flow_from_directory(VAL_DIR,
                                           target_size=IMAGE_SIZE,
                                           batch_size=BATCH_SIZE,
                                           class_mode='categorical')

Found 15 images belonging to 5 classes.
Found 50 images belonging to 5 classes.
Found 9 images belonging to 5 classes.


## Step 4: Model Building: Load PreTrained Model

In [4]:
def get_base_model(name):
    if name == "VGG16":
        return tf.keras.applications.VGG16(weights='imagenet',
                                           include_top=False,
                                           input_shape=(*IMAGE_SIZE, 3))
    elif name == "ResNet50":
        return tf.keras.applications.ResNet50(weights='imagenet',
                                              include_top=False,
                                              input_shape=(*IMAGE_SIZE, 3))
    elif name == "MobileNetV2":
        return tf.keras.applications.MobileNetV2(weights='imagenet',
                                                 include_top=False,
                                                 input_shape=(*IMAGE_SIZE, 3))

base_model = get_base_model(BASE_MODEL_NAME)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


### ====================================================================

# The Most Important 2 Steps in Transfer Learning: 
## 1. FREEZE BASE MODEL

In [5]:
if FREEZE_LAYERS:
    for layer in base_model.layers:
        layer.trainable = False

## 2. ADD CUSTOM HEAD

In [6]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 24,113,541 (91.99 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

### ====================================================================

### Step 4: Model Building Continues..It's COMPILATION Time.

In [7]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Step 5: Model Training | Model Evaluation | Model Testing

In [8]:
history = model.fit(train_data,
                    validation_data=val_data,
                    epochs=EPOCHS)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy: 0.2400 - loss: 1.8333 - val_accuracy: 0.2222 - val_loss: 1.6553
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.1800 - loss: 1.9364 - val_accuracy: 0.2222 - val_loss: 1.6494
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 883ms/step - accuracy: 0.2000 - loss: 2.0403 - val_accuracy: 0.2222 - val_loss: 1.6379
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.1400 - loss: 2.0714 - val_accuracy: 0.2222 - val_loss: 1.6248
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.1400 - loss: 2.0507 - val_accuracy: 0.2222 - val_loss: 1.6219
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 887ms/step - accuracy: 0.1600 - loss: 2.0647 - val_accuracy: 0.2222 - val_loss: 1.6215
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.1800 - loss: 1.9699 - val_accuracy: 0.2222 - val_loss: 1.6240
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.2400 - loss: 1.8873 - val_accuracy: 0.0000e+00 - val_loss: 1.6258
Epoch 

##  (OPTIONAL Step): FINE-TUNING with new weights(NOT SUGGESTED)

In [9]:
if FINE_TUNE_AT is not None:
    for layer in base_model.layers[FINE_TUNE_AT:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print("Starting Fine-Tuning...")

    history_fine = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5
    )

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 25,165,201 (96.00 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 1,051,660 (4.01 MB)

## Export the Intelligence File.

### Question: How to use this file for Prediction?

In [10]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\Vijay\images (4).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predicted Person: DQ


In [11]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\DQ\images (12).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
Predicted Person: DQ


# THE END!